# **Lab 5: Introduction to Web Scraping & Web Crawling**


## Part 1: Concepts

### Web Scraping vs Web Crawling

| Aspect | Web Scraping | Web Crawling |
|--------|-------------|-------------|
| Definition | **Extracting Specific data/content** from web pages | Automated browsing of websites to **find and collect links or pages** |
| Goal | Extract Structure data (e.g., prices, names) | Discover and index web pages |
| Output | Raw or Structured data (CSV, DataFrame) | A list or map of URLs (Site Structure) |
| Tools | BeautifulSoup, Selenium, Puppeteer | Scrapy (Spider), Search engine bots |
| Usage Example | Price Monitoring, data mining, sentiment Analysis | Build dataset for search engine |
| Scope | Narrow - Focuses on content from specific pages | Broad - looks at many pages |
| Frequency | Can be One-Time or Periodic | Often runs continuously (Like Crawlers) |

### Simple Explanation

* Scraping = “Give me this data from this page”
* Crawling = “Explore the website and collect links/data”


## Part 2: Scrape using Beautiful Soup

### Setup

In [76]:
# !pip install requests beautifulsoup4 pandas selenium webdriver-manager


### Import Libraries

In [77]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager


### Define URL

In [78]:
URL_list = "http://quotes.toscrape.com/"

### Add Headers
**Why headers?**

Websites block bots. Headers make your request look like a browser.

In [79]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36"
}

### Send Request

In [80]:
# Configure Selenium options
chrome_options = Options()
chrome_options.add_argument("--headless")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")
chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36")

# Initialize WebDriver
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)

driver.get(URL_list)
print("Status: Page successfully fetched via Selenium. Title:", driver.title)


### Parsing HTML with BeautifulSoup

In [81]:
soup = BeautifulSoup(driver.page_source, "html.parser")
soup


<!DOCTYPE html>

<html lang="en">
<head>
<meta charset="utf-8"/>
<title>Quotes to Scrape</title>
<link href="/static/bootstrap.min.css" rel="stylesheet"/>
<link href="/static/main.css" rel="stylesheet"/>
</head>
<body>
<div class="container">
<div class="row header-box">
<div class="col-md-8">
<h1>
<a href="/" style="text-decoration: none">Quotes to Scrape</a>
</h1>
</div>
<div class="col-md-4">
<p>
<a href="/login">Login</a>
</p>
</div>
</div>
<div class="row">
<div class="col-md-8">
<div class="quote" itemscope="" itemtype="http://schema.org/CreativeWork">
<span class="text" itemprop="text">“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”</span>
<span>by <small class="author" itemprop="author">Albert Einstein</small>
<a href="/author/Albert-Einstein">(about)</a>
</span>
<div class="tags">
            Tags:
            <meta class="keywords" content="change,deep-thoughts,thinking,world" itemprop="keywords"/>
<a class="

#### Extract Page Title

In [82]:
title = soup.find("h1").text.strip()
print(title)

Quotes to Scrape


### Extracting Data

#### Quote Text

In [83]:
quote = soup.find("div", class_="quote")
text = quote.find("span", class_="text").text

print(text)

“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”


#### Quote Author

In [84]:
quote = soup.find("div", class_="quote")
author = quote.find("small", class_="author").text

print(author)

Albert Einstein


#### Quote Tag

In [85]:
quote = soup.find("div", class_="quote")
tags = quote.find("div", class_="tags")
tag = tags.find("a", class_="tag").text

print(tag)

change


### Extract Whole Data

In [86]:
quotes = []

for q in soup.find_all("div", class_="quote"):
    text = q.find("span", class_="text").text
    author = q.find("small", class_="author").text
    tags = q.find("div", class_="tags")
    tag = [t.text for t in tags.find_all("a", class_="tag")]
    quotes.append({"Quote": text, "Author": author, "Tags": tag})

### Store Data in DataFrame

In [87]:
df = pd.DataFrame(quotes)
df.head()

,Quote,Author,Tags
0,“The world as we have created it is a process ...,Albert Einstein,"[change, deep-thoughts, thinking, world]"
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling,"[abilities, choices]"
2,“There are only two ways to live your life. On...,Albert Einstein,"[inspirational, life, live, miracle, miracles]"
3,"“The person, be it gentleman or lady, who has ...",Jane Austen,"[aliteracy, books, classic, humor]"
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe,"[be-yourself, inspirational]"


### Scraping Stocks

In [88]:
URL_LIST = "https://stooq.com/q/?s=aapl.us"

In [89]:
driver.get(URL_LIST)
print("Status: Stock page successfully fetched via Selenium. Title:", driver.title)


In [90]:
soup = BeautifulSoup(driver.page_source, "html.parser")
soup


<html><head><meta content="text/html;charset=utf-8" http-equiv="content-type"/><meta content="apple inc, aapl.us, wykresy, wykres, dane, notowania, gieÅda, rynek" name="keywords"/><meta content="Wykresy notowaÅ spÃ³Åek, indeksÃ³w, kontraktÃ³w, walut, obligacji i towarÃ³w." name="description"/><title>AAPL.US (-0.29%) - Apple Inc - Stooq</title><script>function cpr(a,b){var c=0, f=document.f_c, n=8, s='';if(a){if(a.checked==true){if(b=='wig20'){f.r1.checked=true}else{f.r.value+=(f.r.value?' ':'')+b.toUpperCase()}}else{if(b=='wig20'){f.r1.checked=false}else{b=b.replace('^','\\^');f.r.value=(' '+f.r.value+' ').replace(new RegExp(' '+b+' ','i'),' ');f.r.value=f.r.value.substring(1,f.r.value.length-1)}}}if(f.r.value){t=f.r.value.split(' ');for(i=0;i<t.length;i++){ if(t[i].match('^[A-Za-z0-9._^]{2,16}$') && (!f.r1.checked || t[i]!='wig20')){ if(c++>=n) break; s+=t[i]+'+'; }}s=s.substring(0,s.length-1).toLowerCase();}if(f.r1.checked) s+=(s ? '+' : '')+f.r1.value;location.href='//stooq.com/q

In [91]:
title = soup.find("title").text
print("Title:", title)
clean_title = str(title).split("-")[-2].strip()
print("Clean Title:", clean_title)

Title: AAPL.US (-0.29%) - Apple Inc - Stooq
Clean Title: Apple Inc


In [92]:
tables = soup.find_all("table")
print("Number of tables found:", len(tables))

table = soup.find("table", {"id": "t1"})

rows = table.find_all("tr")
print("Number of rows in the table:", len(rows))

data = {}

for row in rows:
    tds = row.find_all("td")

    for td in tds:
        # 🔹 Extract label (first visible text)
        label = td.get_text(separator="\n").split("\n")[0].strip()

        # 🔹 Extract value from span (if exists)
        span = td.find("span")

        if span:
            value = span.get_text(strip=True)

            # Store only meaningful labels
            if label and value:
                data[label] = value

data

Number of tables found: 57
Number of rows in the table: 21


{'Last': '259.7400',
 'Date': '2026-04-10',
 'Rates of Return': '260.4900',
 'Change': '-0.7500',
 'High/Low': '262.1900',
 '52W Change': '+70.1460',
 'Open': '259.9400',
 'Prev.': '260.4900',
 'Bid': '259.7300',
 'Ask': '259.7400',
 'Volume': '10.7m',
 'Turnover': '2.8g',
 'No. Trades': '59 589'}

In [93]:
date = data.get("Date")
change_value = data.get("52W Change")
volume_value = data.get("Volume")
bid_value = data.get("Bid")
trades_number = data.get("No. Trades")

print("Date:", date)
print("52W Change:", change_value)
print("Volume:", volume_value)
print("Bid:", bid_value)
print("No. of Trades:", trades_number)

Date: 2026-04-10
52W Change: +70.1460
Volume: 10.7m
Bid: 259.7300
No. of Trades: 59 589


In [94]:
df = pd.DataFrame(
    {
        "Title": [clean_title],
        "Date": [date],
        "52W Change": [change_value],
        "Volume": [volume_value],
        "Bid": [bid_value],
        "No. Trades": [trades_number],
    }
)

In [95]:
df

,Title,Date,52W Change,Volume,Bid,No. Trades
0,Apple Inc,2026-04-10,+70.1460,10.7m,259.7300,59 589


# Comparisons between Tools of Scraping

| Feature | BeautifulSoup | Selenium |
|--------|--------------|----------|
| Speed | Fast | Slow |
| JavaScript | No | Yes |
| Browser | No | Yes |
| Difficulty | Easy | Medium |

### When to Use
- Use BeautifulSoup for static pages
- Use Selenium for dynamic pages

In [ ]:
soup_stock = BeautifulSoup(driver.page_source, "html.parser")
print(soup_stock.prettify()[:1000]) # Print first 1000 characters of prettified HTML to avoid huge output


In [ ]:
# Teardown
driver.quit()


In [ ]:
soup_stock = BeautifulSoup(driver.page_source, "html.parser")
print(soup_stock.prettify()[:1000]) # Print first 1000 characters of prettified HTML to avoid huge output


In [ ]:
# Teardown
driver.quit()
